In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [2]:
TRAIN_PATH = "train_log_processed.parquet"
TEST_PATH = "test_log_processed.parquet"

train_df = pq.read_table(TRAIN_PATH).to_pandas()
test_df = pq.read_table(TEST_PATH).to_pandas()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (666085, 61)
Test shape : (166413, 61)


In [3]:
display(train_df.head())
display(test_df.head())

,day_of_week,is_weekend,amount,mcc_code,card_present,device_known,ip_risk_score,is_foreign_txn,time_since_last_s,velocity_1h,...,merchant_country_FR,merchant_country_GB,merchant_country_MX,merchant_country_NG,merchant_country_RO,merchant_country_US,device_type_mobile_app,device_type_phone_ivr,device_type_pos_terminal,device_type_web_browser
0,0.996065,1.575095,-0.371905,-1.464504,-0.735699,0.347452,-0.012692,-0.645055,0.289417,-0.046236,...,False,False,False,False,False,True,True,False,False,False
1,0.996065,1.575095,-0.377483,-0.533962,-0.735699,0.347452,1.167223,1.550255,-0.916995,-0.046236,...,False,False,False,False,False,False,True,False,False,False
2,0.996065,1.575095,1.947426,-2.172881,-0.735699,-2.878092,0.159379,1.550255,-0.800246,-0.960128,...,False,False,False,False,False,False,True,False,False,False
3,0.996065,1.575095,0.532386,2.379672,1.359251,0.347452,0.761627,1.550255,1.568103,-0.960128,...,False,False,False,False,False,False,False,False,True,False
4,0.996065,1.575095,-0.443289,-0.533962,-0.735699,0.347452,1.462202,-0.645055,0.022561,-0.046236,...,False,False,False,False,False,True,True,False,False,False


,day_of_week,is_weekend,amount,mcc_code,card_present,device_known,ip_risk_score,is_foreign_txn,time_since_last_s,velocity_1h,...,merchant_country_FR,merchant_country_GB,merchant_country_MX,merchant_country_NG,merchant_country_RO,merchant_country_US,device_type_mobile_app,device_type_phone_ivr,device_type_pos_terminal,device_type_web_browser
0,-1.501844,-0.634883,-0.358101,-0.297229,-0.735699,-2.878092,-0.012692,-0.645055,0.217143,-0.960128,...,False,False,False,False,False,True,True,False,False,False
1,-1.501844,-0.634883,-0.428539,0.378369,-0.735699,0.347452,0.835372,1.550255,-0.766889,1.781546,...,False,False,False,False,True,False,False,False,False,True
2,-1.501844,-0.634883,-0.275497,0.536798,-0.735699,-2.878092,-0.455161,-0.645055,-0.427759,-0.046236,...,False,False,False,False,False,True,False,False,False,True
3,-1.501844,-0.634883,-0.346597,0.378369,-0.735699,0.347452,-0.565778,-0.645055,1.595900,-0.046236,...,False,False,False,False,False,True,True,False,False,False
4,-1.501844,-0.634883,-0.521196,-0.533962,1.359251,0.347452,-0.651813,-0.645055,0.383929,-0.960128,...,False,False,False,False,False,True,False,True,False,False


In [10]:
print("Số cột train:", len(train_df.columns))
print("Số cột test :", len(test_df.columns))

print("\nTrain columns == Test columns ? ", list(train_df.columns) == list(test_df.columns))

print("\nDtype train:")
print(train_df.dtypes.value_counts())

print("\nDtype test:")
print(test_df.dtypes.value_counts())

Số cột train: 61
Số cột test : 61

Train columns == Test columns ?  True

Dtype train:
float64    33
bool       27
int64       1
Name: count, dtype: int64

Dtype test:
float64    33
bool       27
int64       1
Name: count, dtype: int64


In [11]:
TARGET = "is_fraud"

print("Missing values in train:", train_df.isna().sum().sum())
print("Missing values in test :", test_df.isna().sum().sum())

print("\nTarget distribution - train")
print(train_df[TARGET].value_counts(normalize=True))

print("\nTarget distribution - test")
print(test_df[TARGET].value_counts(normalize=True))

Missing values in train: 0
Missing values in test : 0

Target distribution - train
is_fraud
0    0.982858
1    0.017142
Name: proportion, dtype: float64

Target distribution - test
is_fraud
0    0.982585
1    0.017415
Name: proportion, dtype: float64


In [12]:
X_train = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].astype(int).copy()

X_test = test_df.drop(columns=[TARGET]).copy()
y_test = test_df[TARGET].astype(int).copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)

X_train shape: (666085, 60)
y_train shape: (666085,)
X_test shape : (166413, 60)
y_test shape : (166413,)


In [14]:
bool_cols_train = X_train.select_dtypes(include="bool").columns.tolist()
bool_cols_test = X_test.select_dtypes(include="bool").columns.tolist()

print("Số cột bool trong train:", len(bool_cols_train))
print("Số cột bool trong test :", len(bool_cols_test))

X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
X_test[bool_cols_test] = X_test[bool_cols_test].astype(int)

print("\nDtype sau khi convert:")
print(X_train.dtypes.value_counts())

Số cột bool trong train: 27
Số cột bool trong test : 27

Dtype sau khi convert:
float64    33
int64      27
Name: count, dtype: int64


In [16]:
def evaluate_model(model_name, y_true, y_pred, y_score):
    print(f"========== {model_name} ==========")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
    print("F1-score :", f1_score(y_true, y_pred, zero_division=0))
    print("ROC-AUC  :", roc_auc_score(y_true, y_score))
    print("PR-AUC   :", average_precision_score(y_true, y_score))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

## Train Baseline 1: Logistic Regression

In [22]:
baseline_1 = LogisticRegression(
    max_iter=300,
    solver="liblinear",
    class_weight="balanced",
    random_state=42
)

baseline_1.fit(X_train, y_train)

y_pred_b1 = baseline_1.predict(X_test)
y_score_b1 = baseline_1.predict_proba(X_test)[:, 1]

evaluate_model(
    model_name="Baseline 1 - Logistic Regression",
    y_true=y_test,
    y_pred=y_pred_b1,
    y_score=y_score_b1
)

========== Baseline 1 - Logistic Regression ==========
Accuracy : 0.9523354545618431
Precision: 0.2623229461756374
Recall   : 0.9585921325051759
F1-score : 0.41192170818505336
ROC-AUC  : 0.9925860232313968
PR-AUC   : 0.8453884716492097

Confusion Matrix:
[[155703   7812]
 [   120   2778]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9992    0.9522    0.9752    163515
           1     0.2623    0.9586    0.4119      2898

    accuracy                         0.9523    166413
   macro avg     0.6308    0.9554    0.6935    166413
weighted avg     0.9864    0.9523    0.9654    166413

